## 1. Install necessary libraries

In [1]:
# Install necessary libraries
!pip install --upgrade -q gspread scikit-learn pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 61.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.1 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.1 which is incompatible.
db-dtypes 1.5.0 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.1 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.1 which is incompatible.


##2. Authenticate and access the Google Sheet

In [ ]:
#Google Sheets Linking (Needs user authorization)
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()

gc = gspread.authorize(creds)

In [ ]:
import pandas as pd

# Open the spreadsheet by URL
spreadsheet_url = 'https://docs.google.com/spreadsheets/d/14pMQzQwy2Dl9OIhS24hyJdX1sfEdVi9SeZfGuYaMKQ0/edit?usp=drive_link'
wb = gc.open_by_url(spreadsheet_url)
sheet = wb.get_worksheet(0)

# Load data into DataFrame
data = sheet.get_all_values()
df = pd.DataFrame(data[1:], columns=data[0])

print("Sanity Check - Original Data:")
display(df.head())

Sanity Check - Original Data:


,text,label_specific,label_generic
0,Hi\n\n I am running the IR test...,human_legit,legit
1,\n\n\n\n\n\n\n\n\n\n\n\n\nSecurity Note: Trade...,human_legit,legit
2,\n\n\n\n\n\n\n\n\n\nTrade Me Offer RequestGene...,human_legit,legit
3,"Hi Tony\nNot sure why it didn't work, but I ma...",human_legit,legit
4,Kindly suggest changes\n\n--------------------...,human_legit,legit


## 3. Vectorization (N-grams + TF-IDF)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Vectorization (N-grams + TF-IDF)
# Using unigrams and bigrams (1, 2)
print("Vectorizing text...")
tfidf = TfidfVectorizer(ngram_range=(1, 2), stop_words='english', max_features=10000)
X = tfidf.fit_transform(df['text'].astype(str))

Vectorizing text...


## 4. and 5. Train Model: Adaboost


In [ ]:
# first try
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.ensemble import AdaBoostClassifier
import numpy as np

# Define stratified k-fold cross-validation
skf_strategy = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Define hyperparameter grid for AdaBoost
hyperparam_grid = {
    'n_estimators': [50, 100],  # Number of boosting stages
    'learning_rate': [0.1, 0.5],  # Learning rate
}

# Encode labels numerically
y_generic_encoded = df['label_generic'].apply(lambda x: 1 if x == 'phishing' else 0)

# Instantiate GridSearchCV
grid_search = GridSearchCV(
    estimator=AdaBoostClassifier(random_state=42),  # Uses decision stumps by default
    param_grid=hyperparam_grid,
    cv=skf_strategy, #input stratified cross validation strategy
    scoring={'f1': 'f1', 'roc_auc': 'roc_auc'},
    refit='roc_auc', # Specify which 1 metric to use for finding the best estimator
    verbose=1
)

# Fit the model
print("Training AdaBoost Model (Generic Labels) with Grid Search & Stratified K-Fold...")
grid_search.fit(X, y_generic_encoded)

# View best parameters
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV ROC-AUC score: {grid_search.best_score_:.4f}")

# Use the best estimator as the final model
AdaBoost_Model_generic = grid_search.best_estimator_

Training AdaBoost Model (Generic Labels) with Grid Search & Stratified K-Fold...
Fitting 3 folds for each of 4 candidates, totalling 12 fits
Best parameters: {'learning_rate': 0.5, 'n_estimators': 100}
Best CV ROC-AUC score: 0.9711


In [ ]:
# second try
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.ensemble import AdaBoostClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.multioutput import MultiOutputClassifier
import numpy as np
import pickle

print("MODEL 1: GENERIC LABELS (phishing vs legit)")
print()

# Generic model (binary)
skf_strategy = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

hyperparam_grid = {
    'n_estimators': [50, 100],
    'learning_rate': [0.1, 0.5],
}

y_generic = df['label_generic'].apply(lambda x: 1 if x == 'phishing' else 0)

grid_search = GridSearchCV(
    estimator=AdaBoostClassifier(random_state=42),
    param_grid=hyperparam_grid,
    cv=skf_strategy,
    scoring='roc_auc',
    refit=True,
    verbose=1
)

print("Training Generic Model...")
grid_search.fit(X, y_generic)

print(f"Best params: {grid_search.best_params_}")
print(f"Best ROC-AUC: {grid_search.best_score_:.4f}")

AdaBoost_Model_generic = grid_search.best_estimator_

# Save generic model
with open('adaboost_model_generic.pkl', 'wb') as f:
    pickle.dump(AdaBoost_Model_generic, f)
print("Saved: adaboost_model_generic.pkl")


print("MODEL 2: SPECIFIC LABELS (4 classes)")
print()

# Specific model (multiclass) - SIMPLIFIED!
le = LabelEncoder()
y_specific = le.fit_transform(df['label_specific'])
print(f"Classes: {le.classes_}")

# Same grid search, just different y
grid_search_specific = GridSearchCV(
    estimator=AdaBoostClassifier(random_state=42),
    param_grid=hyperparam_grid,  # Same grid as generic!
    cv=skf_strategy,  # Same CV as generic!
    scoring='accuracy',
    refit=True,
    verbose=1
)

print("Training Specific Model...")
grid_search_specific.fit(X, y_specific)

print(f"Best params: {grid_search_specific.best_params_}")
print(f"Best Accuracy: {grid_search_specific.best_score_:.4f}")

AdaBoost_Model_specific = grid_search_specific.best_estimator_

# Save specific model and encoder
with open('adaboost_model_specific.pkl', 'wb') as f:
    pickle.dump(AdaBoost_Model_specific, f)
print("Saved: adaboost_model_specific.pkl")

with open('label_encoder_specific.pkl', 'wb') as f:
    pickle.dump(le, f)
print("Saved: label_encoder_specific.pkl")

MODEL 1: GENERIC LABELS (phishing vs legit)

Training Generic Model...
Fitting 3 folds for each of 4 candidates, totalling 12 fits
Best params: {'learning_rate': 0.5, 'n_estimators': 100}
Best ROC-AUC: 0.9711
Saved: adaboost_model_generic.pkl
MODEL 2: SPECIFIC LABELS (4 classes)

Classes: ['human_legit' 'human_phishing' 'llm_legit' 'llm_phishing']
Training Specific Model...
Fitting 3 folds for each of 4 candidates, totalling 12 fits
Best params: {'learning_rate': 0.5, 'n_estimators': 100}
Best Accuracy: 0.8095
Saved: adaboost_model_specific.pkl
Saved: label_encoder_specific.pkl


In [ ]:
import pickle

# Save generic model
with open('adaboost_model_generic.pkl', 'wb') as f:
    pickle.dump(AdaBoost_Model_generic, f)
print("AdaBoost_Model_generic saved to adaboost_model_generic.pkl")

# Save specific model
with open('adaboost_model_specific.pkl', 'wb') as f:
    pickle.dump(AdaBoost_Model_specific, f)
print("AdaBoost_Model_specific saved to adaboost_model_specific.pkl")

# Save label encoder for specific model (needed to decode predictions later)
with open('label_encoder_specific.pkl', 'wb') as f:
    pickle.dump(le, f)
print("Label encoder saved to label_encoder_specific.pkl")

AdaBoost_Model_generic saved to adaboost_model_generic.pkl
AdaBoost_Model_specific saved to adaboost_model_specific.pkl
Label encoder saved to label_encoder_specific.pkl


# 6. Evaluation

In [ ]:
# 6a. Preprocess Test Data - Use same TFIDF as train set
from sklearn.feature_extraction.text import TfidfVectorizer

print("EVALUATING BOTH MODELS")
print()

print("Ensuring TF-IDF vectorizer is fitted on training data...")
tfidf = TfidfVectorizer(ngram_range=(1, 2), stop_words='english', max_features=10000)
X = tfidf.fit_transform(df['text'].astype(str))
print("TF-IDF vectorizer fitted. Training data shape:", X.shape)

print("Transforming evaluation data...")
X_test = tfidf.transform(df['text'].astype(str))
print("X_test created with shape:", X_test.shape)

EVALUATING BOTH MODELS

Ensuring TF-IDF vectorizer is fitted on training data...
TF-IDF vectorizer fitted. Training data shape: (4000, 10000)
Transforming evaluation data...
X_test created with shape: (4000, 10000)


In [ ]:
# 6b. Define Custom Cost Function
import numpy as np

def calculate_custom_cost(y_true, y_pred):
    """
    Calculates a custom cost based on False Negatives and False Positives.
    Cost = (15 * False Negatives) + False Positives
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    false_negatives = np.sum((y_true == 1) & (y_pred == 0))
    false_positives = np.sum((y_true == 0) & (y_pred == 1))

    custom_cost = (15 * false_negatives) + false_positives
    return custom_cost

print("Function 'calculate_custom_cost' defined.")

Function 'calculate_custom_cost' defined.


In [ ]:
# 6c. Encode Test Labels
# Generic labels
y_true_generic = df['label_generic'].apply(lambda x: 1 if x == 'phishing' else 0)
print(f"\ny_true_generic shape: {y_true_generic.shape}")

# Specific labels - load the same encoder used in training
import pickle
with open('label_encoder_specific.pkl', 'rb') as f:
    le = pickle.load(f)
y_true_specific = le.transform(df['label_specific'])
print(f"y_true_specific shape: {y_true_specific.shape}")
print(f"Classes: {le.classes_}")


y_true_generic shape: (4000,)
y_true_specific shape: (4000,)
Classes: ['human_legit' 'human_phishing' 'llm_legit' 'llm_phishing']


In [ ]:
# 6d. Load Pre-Trained Models
with open('adaboost_model_generic.pkl', 'rb') as f:
    AdaBoost_Model_generic = pickle.load(f)
print("\nAdaBoost_Model_generic loaded successfully.")

with open('adaboost_model_specific.pkl', 'rb') as f:
    AdaBoost_Model_specific = pickle.load(f)
print("AdaBoost_Model_specific loaded successfully.")


AdaBoost_Model_generic loaded successfully.
AdaBoost_Model_specific loaded successfully.


In [ ]:
# 6e. Get Predictions and Probabilities
print("\nMaking predictions with AdaBoost_Model_generic...")
y_pred_generic = AdaBoost_Model_generic.predict(X_test)
y_proba_generic = AdaBoost_Model_generic.predict_proba(X_test)[:, 1]
print(f"Generic predictions shape: {y_pred_generic.shape}")

print("Making predictions with AdaBoost_Model_specific...")
y_pred_specific = AdaBoost_Model_specific.predict(X_test)
y_proba_specific = AdaBoost_Model_specific.predict_proba(X_test)
print(f"Specific predictions shape: {y_pred_specific.shape}")


Making predictions with AdaBoost_Model_generic...
Generic predictions shape: (4000,)
Making predictions with AdaBoost_Model_specific...
Specific predictions shape: (4000,)


In [ ]:
# 6f. Create Metrics
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, classification_report, confusion_matrix

# Generic Model Evaluation
print("MODEL 1: GENERIC LABELS")
print()

cm = confusion_matrix(y_true_generic, y_pred_generic)
tn, fp, fn, tp = cm.ravel()

print(f"Accuracy: {accuracy_score(y_true_generic, y_pred_generic):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_true_generic, y_proba_generic):.4f}")
print(f"Cost (15*FN+FP): {(15*fn)+fp}")
print(f"\nConfusion Matrix:")
print(f"[[{tn:4d}  {fp:4d}]\n [{fn:4d}  {tp:4d}]]")

# Specific Model Evaluation
print("MODEL 2: SPECIFIC LABELS")
print()

print(f"Accuracy: {accuracy_score(y_true_specific, y_pred_specific):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_true_specific, y_pred_specific,
                          target_names=le.classes_))
print(f"\nConfusion Matrix:\n{confusion_matrix(y_true_specific, y_pred_specific)}")

MODEL 1: GENERIC LABELS

Accuracy: 0.9177
ROC-AUC: 0.9719
Cost (15*FN+FP): 2009

Confusion Matrix:
[[1791   209]
 [ 120  1880]]
MODEL 2: SPECIFIC LABELS

Accuracy: 0.8153

Classification Report:
                precision    recall  f1-score   support

   human_legit       0.96      0.85      0.90      1000
human_phishing       0.83      0.93      0.88      1000
     llm_legit       0.83      0.66      0.74      1000
  llm_phishing       0.68      0.81      0.74      1000

      accuracy                           0.82      4000
     macro avg       0.83      0.82      0.82      4000
  weighted avg       0.83      0.82      0.82      4000


Confusion Matrix:
[[852 142   3   3]
 [ 26 933   3  38]
 [  0   3 663 334]
 [  8  45 134 813]]


## Visualization

In [ ]:
import plotly.express as px
import pandas as pd
from sklearn.metrics import classification_report

report = classification_report(y_true_specific, y_pred_specific,
                             target_names=le.classes_, output_dict=True)

df = pd.DataFrame({
    'Category': le.classes_,
    'Accuracy': [report[cat]['precision'] for cat in le.classes_]
})

px.bar(df, x='Category', y='Accuracy',
       title='Accuracy by Email Type',
       color='Category').show()